In [ ]:
from google.colab import drive
drive.mount('/content/drive/')

Mounted at /content/drive/


In [ ]:
import pandas as pd
import os
import random

import numpy as np
from sklearn.preprocessing import StandardScaler
import geopandas as gpd

from spreg import GM_Combo, gets_sdm, GM_Lag
import libpysal
from spreg import OLS
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics.pairwise import rbf_kernel
from esda.moran import Moran
from sklearn.feature_selection import mutual_info_classif
from esda.moran import Moran
import libpysal
# !pip install torchmetrics
# from torchmetrics.functional.clustering import mutual_info_score

In [ ]:
# Navigate to your project root (adjust this path as needed)
# After mounting, your Drive files will be in /content/drive/MyDrive
# You should set project_root to the specific folder within MyDrive where your project resides.
# This path *can* contain spaces, as you're navigating within the already mounted Drive.
project_root = '/content/drive/Othercomputers/My Mac/01_GeoTrans/Geo_Domain_Shift'

# Change the current working directory if you want relative paths to work from there
os.chdir(project_root)

print(f"Current working directory after changing: {os.getcwd()}")

# Now you can access your data, e.g., if it's in a 'data' folder within your project_root
# data_path = os.path.join(project_root, 'data')
# print(f"Path to data: {data_path}")

# If you just want to access it directly, without changing CWD:
# data_folder = '/content/drive/MyDrive/path/to/your/data'
# print(f"Data folder: {data_folder}")

Current working directory after changing: /content/drive/Othercomputers/My Mac/01_GeoTrans/Geo_Domain_Shift


In [ ]:
def load_all_areas(if_shuffle=True):
    areas = os.listdir("data")
    if if_shuffle:
        random.shuffle(areas)
    return areas

def split_train_valid_test(areas, train_ratio=0.8, valid_ratio=0.1, test_ratio=0.1):
    assert train_ratio + valid_ratio + test_ratio == 1

    train_areas = areas[:int(len(areas)*train_ratio)]
    valid_areas = areas[int(len(areas)*train_ratio):int(len(areas)*(train_ratio+valid_ratio))]
    test_areas = areas[int(len(areas)*(train_ratio+valid_ratio)):]
    return train_areas, valid_areas, test_areas


def construct_train(areas):

    xs = []
    ys = []
    for area in areas:
        if area == ".DS_Store":
            continue
        demos = np.load(f"data/{area}/demos.npy")
        pois = np.load(f"data/{area}/pois.npy")

        dis = np.load(f"data/{area}/dis.npy")


        feat = np.concatenate([demos, pois], axis=1)

        feat_o, feat_d = feat, feat
        feat_o = feat_o.reshape([feat_o.shape[0], 1, feat_o.shape[1]]).repeat(feat_o.shape[0], axis=1)
        feat_d = feat_d.reshape([1, feat_d.shape[0], feat_d.shape[1]]).repeat(feat_d.shape[0], axis=0)
        dis = dis.reshape([dis.shape[0], dis.shape[1], 1])

        x = np.concatenate([feat_o, feat_d, dis], axis=2)
        x = x.reshape([-1, x.shape[2]])
        xs.append(x)
        # x = list(np.expand_dims(x, axis = 1))
        # xs.extend(x)

        od = np.load(f"data/{area}/od.npy")

        # divede by total out flow
        od = od/(od.sum(1)[:, None] +  + 1e-12)
        y = od.reshape([-1])
        ys.append(y)
        # y = list(np.expand_dims(od, axis = 1))
        # ys.extend(y)

    x = np.concatenate(xs, axis=0)
    y = np.concatenate(ys, axis=0)

    return x, y


def construct_validtest(areas):
    x_areas = []
    y_areas = []
    for area in areas:
        if area == ".DS_Store":
            continue
        demos = np.load(f"data/{area}/demos.npy")
        pois = np.load(f"data/{area}/pois.npy")

        dis = np.load(f"data/{area}/dis.npy")


        feat = np.concatenate([demos, pois], axis=1)

        feat_o, feat_d = feat, feat
        feat_o = feat_o.reshape([feat_o.shape[0], 1, feat_o.shape[1]]).repeat(feat_o.shape[0], axis=1)
        feat_d = feat_d.reshape([1, feat_d.shape[0], feat_d.shape[1]]).repeat(feat_d.shape[0], axis=0)
        dis = dis.reshape([dis.shape[0], dis.shape[1], 1])

        x = np.concatenate([feat_o, feat_d, dis], axis=2)
        x = x.reshape([-1, x.shape[2]])
        x_areas.append(x)

        od = np.load(f"data/{area}/od.npy")
        y = od.reshape([-1])

        y_areas.append(y)

    return x_areas, y_areas

def load_data(if_shuffle=True):
    areas = load_all_areas(if_shuffle)
    train_areas, valid_areas, test_areas = split_train_valid_test(areas)

    x_train, y_train = construct_train(train_areas)
    x_valid, y_valid = construct_validtest(valid_areas)
    x_test, y_test = construct_validtest(test_areas)

    return x_train, y_train, x_valid, y_valid, x_test, y_test

def load_state_areas(if_shuffle=True, state_codes:list = None):
    # areas = os.listdir("data")
    areas = [area for area in os.listdir("data") if area.startswith(tuple(state_codes))]
    if if_shuffle:
        random.shuffle(areas)
    return areas

def load_data_by_states(if_shuffle=True, train_state_codes:list = None, test_state_codes:list = None ):
    areas = load_state_areas(if_shuffle, train_state_codes)
    train_areas, valid_areas, _ = split_train_valid_test(areas, train_ratio=0.7, valid_ratio=0.3, test_ratio=0.0)

    x_train, y_train = construct_train(train_areas)
    x_valid, y_valid = construct_validtest(valid_areas)

    areas = load_state_areas(if_shuffle, test_state_codes)
    _, _, test_areas = split_train_valid_test(areas, train_ratio=0.0, valid_ratio=0.0, test_ratio=1.0)
    x_test, y_test = construct_validtest(test_areas)

    return x_train, y_train, x_valid, y_valid, x_test, y_test, test_areas

In [ ]:
def multivariate_morans_I(X, W):
    """
    Compute multivariate Moran's I for high-dimensional features.

    Parameters
    ----------
    X : array-like, shape (n, p)
        Feature matrix, rows = regions, columns = features.
    W : libpysal.weights.W
        Spatial weights object.

    Returns
    -------
    I_mv : float
        Multivariate Moran's I.
    """
    X = np.asarray(X, dtype=float)
    n, p = X.shape

    # Center features (multivariate mean)
    mean_vec = X.mean(axis=0, keepdims=True)
    Z = X - mean_vec  # (n, p)

    # Convert weights to full matrix (or use sparse for large n)
    Wmat = W.full()[0]  # (n, n)
    S0 = Wmat.sum()

    # Numerator: sum_{ij} w_ij * z_i^T z_j
    # Efficient: (WZ) elementwise times Z, summed over all entries
    WZ = Wmat @ Z               # (n, p)
    num = np.sum(WZ * Z)        # scalar

    # Denominator: sum_i ||z_i||^2
    den = np.sum(Z**2)          # scalar

    I_mv = (n / S0) * (num / den)
    return I_mv

In [ ]:
from math import dist
def load_area_Morans(areas):
    morans_results = [] # To store Moran's I values for each feature of each area
    feats = []
    # cloeness_results = []
    for area in areas:
        if area == ".DS_Store":
            continue
        demos = np.load(f"data/{area}/demos.npy")
        pois = np.load(f"data/{area}/pois.npy")
        dis = np.load(f"data/{area}/dis.npy")

        feat = np.concatenate([demos, pois], axis=1)
        feats.append(feat)

        # --- Convert distance matrix to weights ---
        np.fill_diagonal(dis, np.inf)
        W_matrix = 1 / dis

        # Make sure no inf/nan remain:
        W_matrix[~np.isfinite(W_matrix)] = 0.0

        # Convert to PySAL weights object
        w = libpysal.weights.full2W(W_matrix)

        # Standardize weights (row-standardization)
        w.transform = 'r'

        area_morans_I = multivariate_morans_I(feat, w)

        morans_results.append(area_morans_I) # Append list of Moran's I for all features in this area

    feats = np.concatenate(feats, axis=0)
    morans = np.array(morans_results)

    return feats, morans

In [ ]:
us_county_geo_file = 'geo_data/tl_2018_us_county/tl_2018_us_county.shp'
gpd_us_county =  gpd.read_file(us_county_geo_file)
gpd_us_county = gpd_us_county [~gpd_us_county['STATEFP'].isin(['02','15','60','66','69','72','78'])]

In [ ]:
def domain_shift_mi(src_areas, tar_areas):
    rows = []
    src_feats, src_morans = load_area_Morans(src_areas)

    gpd_src_areas = gpd_us_county[gpd_us_county['GEOID'].isin(src_areas)].to_crs(epsg=5070)

    src_morans = np.nan_to_num(src_morans, nan=0.0)

    for area in tar_areas:
        if area == ".DS_Store":
            continue

        # Check if the target area exists in gpd_us_county after filtering
        gpd_tar_area = gpd_us_county[gpd_us_county['GEOID'] == area]
        if gpd_tar_area.empty:
            # Skip this area if it's not found in the filtered gpd_us_county
            print(f"Warning: GEOID {area} not found in gpd_us_county. Skipping.")
            continue

        gpd_tar_area = gpd_tar_area.to_crs(epsg=5070)

        tar_feats, tar_moran = load_area_Morans([area])
        tar_moran = np.nan_to_num(tar_moran, nan=0.0)

        # mutual information shift
        x_one = np.asarray(tar_feats)

        # combine source + this domain
        X = np.vstack([src_feats, x_one])

        # domain labels (0 = train, 1 = target domain i)
        D = np.hstack([np.zeros(len(src_feats)), np.ones(len(x_one))])

        # MI per feature (length = n_features)
        mi = mutual_info_classif(X, D, discrete_features=False)
        mi_shift = mi.mean(axis=0)

        # Moran's I shift
        mu_s = src_morans.mean()
        sigma_s = src_morans.std()
        moran_shift = (tar_moran - mu_s) / (sigma_s + 1e-12)
        moran_shift = moran_shift.item()

        row = {'src_state':gpd_src_areas.iloc[0]['STATEFP'], 'tar_geoid': area, 'mi_shift': mi_shift}

        rows.append(row)

    df_domain_shift = pd.DataFrame(rows)

    return df_domain_shift

In [ ]:
train_states = ['01']
test_states = ['06']

In [ ]:
train_areas = load_state_areas(if_shuffle=False, state_codes = train_states)

In [ ]:
test_areas = load_state_areas(if_shuffle=False, state_codes = test_states)

In [ ]:
df_domain_shift = domain_shift(train_areas, test_areas)

In [ ]:
df_domain_shift

,src_state,tar_geoid,mi_shift,moran_shift
0,01,06001,0.025831,1.175199
1,01,06005,0.002037,0.690554
2,01,06007,0.004499,1.251975
3,01,06009,0.002587,0.563915
4,01,06011,0.001067,-1.853494
5,01,06013,0.014350,1.201559
6,01,06015,0.001161,0.085667
7,01,06017,0.004972,0.895543
8,01,06019,0.017252,1.150467
9,01,06021,0.001008,-0.284583


### For all source states

In [ ]:
df_us_states = pd.read_csv("geo_data/us_states_fips.csv", dtype={"FIPS": str})

In [ ]:
def domain_shift_moran(src_areas, tar_areas):
    rows = []
    src_feats, src_morans = load_area_Morans(src_areas)

    gpd_src_areas = gpd_us_county[gpd_us_county['GEOID'].isin(src_areas)].to_crs(epsg=5070)

    src_morans = np.nan_to_num(src_morans, nan=0.0)

    for area in tar_areas:
        if area == ".DS_Store":
            continue

        # Check if the target area exists in gpd_us_county after filtering
        gpd_tar_area = gpd_us_county[gpd_us_county['GEOID'] == area]
        if gpd_tar_area.empty:
            # Skip this area if it's not found in the filtered gpd_us_county
            print(f"Warning: GEOID {area} not found in gpd_us_county. Skipping.")
            continue

        gpd_tar_area = gpd_tar_area.to_crs(epsg=5070)

        tar_feats, tar_moran = load_area_Morans([area])
        tar_moran = np.nan_to_num(tar_moran, nan=0.0)

        # mutual information shift
        # x_one = np.asarray(tar_feats)

        mu_s = src_morans.mean()
        sigma_s = src_morans.std()
        moran_shift = (tar_moran - mu_s) / (sigma_s + 1e-12)
        moran_shift = moran_shift.item()

        # 'src_moran': src_morans, 'tar_moran':tar_moran,
        row = {'src_state':gpd_src_areas.iloc[0]['STATEFP'], 'tar_geoid': area, 'src_moran': src_morans, 'tar_moran':tar_moran, 'moran_shift': moran_shift}

        rows.append(row)

    df_domain_shift = pd.DataFrame(rows)

    return df_domain_shift

In [ ]:
source_states = [['01'], ['04'], ['05'], ['06'], ['08'], ['09'],
 ['10'], ['12'], ['13'], ['16'], ['17'], ['18'], ['19'],
  ['20'], ['21'], ['22'], ['23'],['24'],['25'],['26'],['27'],['28'],['29'],
  ['30'],['31'],['32'],['33'],['34'],['35'],['36'],['37'],['38'],['39'],
  ['40'], ['41'], ['42'], ['44'],['45'],['46'],['47'],['48'],['49'],
  ['50'], ['51'], ['53'], ['54'], ['55'], ['56']]

In [ ]:
for train_states in source_states:
    src_areas = load_state_areas(if_shuffle=False, state_codes = train_states)
    df_us_states = df_us_states[~df_us_states['FIPS'].isin(['02','15','60','66','69','72','78'])]

    test_states = list(set(df_us_states['FIPS']) - set(train_states))
    test_areas = load_state_areas(if_shuffle=False, state_codes = test_states)

    df_domain_shift = domain_shift_moran(src_areas, test_areas)

    df_domain_shift.to_csv('models/DGM_raw/evaluations/domain_shift/by_source_state_moran/{}_domain_shift.csv'.format('+'.join(train_states)), index=False)

KeyboardInterrupt: 